In [ ]:
# ============================================
# SILVER LAYER - SALES CLEANING
# ============================================

import logging
from pyspark.sql.functions import col, when, to_timestamp
from pyspark.sql.types import IntegerType, DoubleType
from utils.validation import validate_sales

# ============================================
# CONFIG
# ============================================

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================
# LOAD DATA
# ============================================

df = spark.read.parquet("Files/Bronze/processed/sales_raw")
logger.info("Loaded sales raw data")

# ============================================
# TRANSFORMATION
# ============================================

df = df.withColumn("customer_age", col("customer_age").cast(IntegerType())) \
       .withColumn("total_amount", col("total_amount").cast(DoubleType())) \
       .withColumn("feedback_score", col("feedback_score").cast(IntegerType()))

df = df.withColumn("order_date", to_timestamp(col("order_date")))

# Business logic: refund → negative
df = df.withColumn(
    "final_amount",
    when(col("order_status") == "Refunded", -col("total_amount"))
    .otherwise(col("total_amount"))
)

# ============================================
# VALIDATION
# ============================================

valid_df, invalid_df = validate_sales(df)

# ============================================
# WRITE OUTPUT
# ============================================

valid_df.write.mode("overwrite").saveAsTable("silver_sales_clean")
invalid_df.write.mode("overwrite").saveAsTable("silver_sales_quarantine")

logger.info(f"Valid rows: {valid_df.count()}")
logger.info(f"Invalid rows: {invalid_df.count()}")

# Optional preview
print("Sample cleaned sales:")
spark.read.table("silver_sales_clean").limit(5).show()

# ============================================
# EXIT
# ============================================

mssparkutils.notebook.exit("Success")